# Live Product Category Detection (YOLO-World + Gradio)

This notebook is a no-Supabase live webcam prototype for the 17 requested categories. It draws **real model-generated bounding boxes**, not a fixed overlay. It uses YOLO-World zero-shot detection so it can run without custom box annotations. Zero-shot detections are a starting point, not a guarantee of perfect accuracy; for reliable custom categories such as biscuit packets and chips packets, fine-tune on a labeled object-detection dataset.

**Note:** Ultralytics YOLO-World uses PyTorch, not TensorFlow. Your original notebook's EfficientNet CNN is an image classifier and cannot generate object boxes. This notebook uses YOLO-World for the live detection requirement and does not connect to Supabase.

## 1. Install dependencies
Run once after selecting a GPU runtime (Runtime → Change runtime type → T4 GPU, if available).

In [ ]:
%pip -q install -U ultralytics gradio opencv-python-headless

## 2. Load YOLO-World and define the allowed categories
The prompt descriptions help the open-vocabulary detector look for each object. Only these 17 labels are shown in the output.

In [ ]:
import cv2
import numpy as np
import gradio as gr
import torch
from ultralytics import YOLO

CATEGORIES = [
    'biscuit', 'chips', 'pen', 'scale', 'phone', 'calculator',
    'headphone', 'laptop', 'books', 'pencils', 'erasers', 'keyboard',
    'mouse', 'toothpaste', 'ball', 'toothbrush', 'wires'
]

# Text prompts are used by YOLO-World to localize objects.
# Output labels are mapped back to the short category names above.
PROMPTS = [
    'biscuit packet or cookie package',
    'chips packet or potato chips bag',
    'pen or ballpoint pen',
    'ruler or measuring scale',
    'mobile phone or smartphone',
    'calculator',
    'headphones or headset',
    'laptop computer',
    'book or books',
    'pencil or pencils',
    'eraser or rubber',
    'computer keyboard',
    'computer mouse',
    'toothpaste tube',
    'ball',
    'toothbrush',
    'wire or cable'
]

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Device:', 'GPU' if DEVICE == 0 else 'CPU')
print('Loading YOLO-World weights; the first run downloads the model...')
detector = YOLO('yolov8s-worldv2.pt')
detector.set_classes(PROMPTS)
print('Detector ready. Allowed labels:', ', '.join(CATEGORIES))

## 3. Real object detection and green bounding boxes
The detector processes the live frame, then draws a green box around each detected object. Confidence threshold can be lowered slightly if detections are being missed, but too low a value can increase false detections.

In [ ]:
CONFIDENCE = 0.20
IMAGE_SIZE = 416  # Smaller input is faster; increase to 640 for small objects.

def detect_frame(frame):
    if frame is None:
        return None

    # Gradio provides RGB arrays. Ultralytics accepts numpy image arrays.
    frame = np.asarray(frame)
    if frame.ndim != 3 or frame.shape[2] < 3:
        return frame
    frame = frame[:, :, :3].copy()

    try:
        results = detector.predict(
            source=frame,
            imgsz=IMAGE_SIZE,
            conf=CONFIDENCE,
            device=DEVICE,
            verbose=False,
            max_det=20
        )
        result = results[0]
        output = frame.copy()
        if result.boxes is None:
            return output

        height, width = output.shape[:2]
        for box in result.boxes:
            class_id = int(box.cls.item())
            confidence = float(box.conf.item())
            if class_id < 0 or class_id >= len(CATEGORIES):
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            x1, x2 = max(0, min(x1, width - 1)), max(0, min(x2, width - 1))
            y1, y2 = max(0, min(y1, height - 1)), max(0, min(y2, height - 1))
            label = f'{CATEGORIES[class_id].upper()} {confidence:.0%}'
            cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 3)
            font = cv2.FONT_HERSHEY_SIMPLEX
            scale, thickness = 0.65, 2
            (tw, th), baseline = cv2.getTextSize(label, font, scale, thickness)
            top = max(0, y1 - th - baseline - 8)
            cv2.rectangle(output, (x1, top), (min(width - 1, x1 + tw + 10), y1), (0, 255, 0), -1)
            cv2.putText(output, label, (x1 + 5, max(th + 2, y1 - baseline - 4)), font, scale, (0, 0, 0), thickness, cv2.LINE_AA)
        return output
    except Exception as exc:
        print('Detection frame error:', repr(exc))
        return frame

# Quick smoke test without a webcam: the function should accept an image array.
print('Detection function ready. Start the webcam in the next cell.')

## 4. Launch the live Gradio webcam filter
Open the public Gradio link, allow camera access, and hold one item clearly in view. In Colab, keep this cell running while testing.

In [ ]:
with gr.Blocks(title='Live Product Category Detection') as demo:
    gr.Markdown('# 🟢 Live Product Category Detection')
    gr.Markdown('Live YOLO-World detection with real green bounding boxes. Supported labels: biscuit, chips, pen, scale, phone, calculator, headphone, laptop, books, pencils, erasers, keyboard, mouse, toothpaste, ball, toothbrush, wires.')
    with gr.Row():
        webcam = gr.Image(sources=['webcam'], type='numpy', streaming=True, label='Live camera')
        output = gr.Image(type='numpy', label='Live detections')
    webcam.stream(
        fn=detect_frame,
        inputs=webcam,
        outputs=output,
        stream_every=0.15,
        concurrency_limit=1,
        time_limit=3600
    )

demo.queue().launch(share=True, debug=False)

## 5. Accuracy notes and next steps

- This is a zero-shot live detector. It can draw boxes immediately without training, but it may confuse visually similar objects or miss product packaging.
- Your old EfficientNet model classifies a whole image and its image folders do not contain bounding-box annotations. Those images alone cannot directly train a YOLO detector to draw accurate boxes.
- For a dependable custom detector, collect varied photos for all 17 categories and label each object with a bounding box (for example, in Roboflow), then fine-tune YOLO on the exported YOLO-format dataset.
- This notebook does not use OCR, Supabase, inventory updates, or database credentials.